# 埋め込みで意味検索 (セマンティックサーチ) をする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_similarity_search/semantic_search.ipynb)

NLP クックブック 12-02 ／ 文埋め込み・類似度・検索

キーワードが一致しなくても、質問と意味が近い文書を検索したい。

In [1]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("intfloat/multilingual-e5-small")

docs = [
    "富士山は標高 3776 メートルで、日本で最も高い山である。",
    "東京スカイツリーは高さ 634 メートルの電波塔である。",
    "琵琶湖は滋賀県にある日本最大の湖である。",
    "信濃川は全長 367 キロメートルで、日本で最も長い川である。",
    "形態素解析とは、文を単語に分割して品詞を付与する処理である。",
    "Transformer は自己注意機構に基づくニューラルネットワークである。",
    "カレーライスは明治時代に日本に伝わった料理である。",
]
doc_emb = model.encode(["passage: " + d for d in docs], normalize_embeddings=True, convert_to_tensor=True)

def search(query, k=3):
    q = model.encode("query: " + query, normalize_embeddings=True, convert_to_tensor=True)
    for hit in util.semantic_search(q, doc_emb, top_k=k)[0]:
        print(f"  {hit['score']:.3f}  {docs[hit['corpus_id']]}")

for q in ["日本一大きい湖はどこ？", "単語に区切る技術", "What is the tallest mountain in Japan?"]:
    print(q)
    search(q)

日本一大きい湖はどこ？
  0.842  琵琶湖は滋賀県にある日本最大の湖である。
  0.780  信濃川は全長 367 キロメートルで、日本で最も長い川である。
  0.774  富士山は標高 3776 メートルで、日本で最も高い山である。
単語に区切る技術
  0.828  形態素解析とは、文を単語に分割して品詞を付与する処理である。
  0.796  Transformer は自己注意機構に基づくニューラルネットワークである。
  0.740  東京スカイツリーは高さ 634 メートルの電波塔である。
What is the tallest mountain in Japan?
  0.771  富士山は標高 3776 メートルで、日本で最も高い山である。
  0.765  東京スカイツリーは高さ 634 メートルの電波塔である。
  0.737  信濃川は全長 367 キロメートルで、日本で最も長い川である。


- 文書側の埋め込みは 1 回計算して保存しておき、検索時はクエリだけ埋め込む。
- 数十万件までなら総当たり (この方法) で十分速い。それ以上は近似最近傍探索 (次項の FAISS) を使う。